# Base Model vs SFT Model — Comparison

Evaluates `Qwen/Qwen2.5-Coder-3B-Instruct` (base) against the LoRA SFT adapter (`Sizzing/aws-rl-sft-qwen25coder3b-adapter`) on the AWS RL eval set.

**Metrics compared**
| Metric | What it measures |
|---|---|
| `format%` | Response starts with `aws ` verbatim |
| `format_after_extract%` | Starts with `aws ` after stripping fences/prose |
| `exact%` | Extracted command matches canonical exactly |
| `service%` | Same AWS service (e.g. both `aws s3api`) |
| `operation%` | Same service AND operation (e.g. both `aws s3api create-bucket`) |
| `avg_latency` | Mean inference time per prompt (seconds) |
| `avg_len` | Mean response character length |

**Before running:**
1. Runtime → Change runtime type → GPU (T4)
2. Fill in Config below (HF token + optional adapter repo)

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
BASE_MODEL        = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
SFT_ADAPTER_REPO  = "Sizzing/aws-rl-sft-qwen25coder3b-adapter"  # HF Hub adapter
DATASET_REPO      = "Sizzing/aws-rl-sft"
MAX_SEQ_LENGTH    = 512
MAX_NEW_TOKENS    = 120
EVAL_MAX_PER_COMBO = 2   # rows per (difficulty, source) combo in the eval set
IS_COLAB          = True  # set False if running locally
# ───────────────────────────────────────────────────────────────────────────
print("Config OK")

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth
!pip install -q --force-reinstall --no-deps "transformers>=4.50,<5.0"
!pip install -q --upgrade "trl<0.12.0" peft accelerate datasets huggingface_hub bitsandbytes
!pip install -q matplotlib numpy

In [ ]:
import os, gc, time, json
import torch

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

assert torch.cuda.is_available(), "No GPU — Runtime → Change runtime type → GPU"
gpu = torch.cuda.get_device_properties(0)
IS_T4   = "T4" in gpu.name
USE_FP16 = IS_T4
USE_BF16 = not IS_T4
print(f"GPU  : {gpu.name}")
print(f"VRAM : {gpu.total_memory / 1e9:.1f} GB")
print(f"Prec : {'fp16' if USE_FP16 else 'bf16'}")

In [ ]:
if IS_COLAB:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets (or export locally)"

from huggingface_hub import login as hf_login
hf_login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF authenticated")

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_REPO, token=os.environ["HF_TOKEN"])
print(ds)

## Eval harness

Identical scoring function used twice — once on the base model, once on the SFT model — so the comparison is apples-to-apples.

In [ ]:
def extract_command(raw: str) -> str:
    text = raw.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(l for l in lines if not l.startswith("```")).strip()
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("aws "):
            return line
    return text


def score_row(completion: str, expected: str) -> dict:
    extracted  = extract_command(completion)
    e_tokens   = extracted.split()
    exp_tokens = expected.split()
    return {
        "format_ok":            completion.strip().startswith("aws "),
        "format_after_extract": extracted.startswith("aws "),
        "exact":                extracted == expected.strip(),
        "service":              (len(e_tokens) >= 2 and len(exp_tokens) >= 2
                                 and e_tokens[1:2] == exp_tokens[1:2]),
        "operation":            (len(e_tokens) >= 3 and len(exp_tokens) >= 3
                                 and e_tokens[2:3] == exp_tokens[2:3]),
    }


def build_eval_set(dataset, max_per_combo: int = 2):
    seen, picks = {}, []
    for r in dataset:
        key = (r["difficulty"], r["source"])
        seen[key] = seen.get(key, 0) + 1
        if seen[key] <= max_per_combo:
            picks.append(r)
    return picks


def eval_model(model, tokenizer, eval_set, max_new_tokens: int = 120) -> dict:
    results = []
    model.eval()
    for row in eval_set:
        msgs     = row["messages"][:2]          # system + user
        expected = row["messages"][2]["content"]
        prompt   = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        t0 = time.time()
        with torch.inference_mode():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        dt         = time.time() - t0
        completion = tokenizer.decode(
            out_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True
        )
        s = score_row(completion, expected)
        s.update({"latency": dt, "len": len(completion),
                  "completion": completion, "expected": expected})
        results.append(s)

    n = len(results)
    return {
        "format_pct":               sum(r["format_ok"]            for r in results) / n,
        "format_after_extract_pct": sum(r["format_after_extract"] for r in results) / n,
        "exact_pct":                sum(r["exact"]                for r in results) / n,
        "service_pct":              sum(r["service"]              for r in results) / n,
        "operation_pct":            sum(r["operation"]            for r in results) / n,
        "avg_latency":              sum(r["latency"]              for r in results) / n,
        "avg_len":                  sum(r["len"]                  for r in results) / n,
        "_per_row":                 results,
    }


EVAL_SET = build_eval_set(ds["validation"], max_per_combo=EVAL_MAX_PER_COMBO)
combos   = len(set((r["difficulty"], r["source"]) for r in EVAL_SET))
print(f"Eval set: {len(EVAL_SET)} prompts across {combos} (tier, source) combos")

## Step 1 — Evaluate base model

In [ ]:
from unsloth import FastLanguageModel

print("Loading base model...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)

print("Running base model eval...")
base_metrics = eval_model(base_model, base_tokenizer, EVAL_SET,
                          max_new_tokens=MAX_NEW_TOKENS)

print("\n=== BASE MODEL ===")
for k, v in base_metrics.items():
    if k == "_per_row":
        continue
    print(f"  {k:<30} {100*v:6.1f}%" if "pct" in k else f"  {k:<30} {v:.3f}")

del base_model, base_tokenizer
gc.collect(); torch.cuda.empty_cache()
print("\nBase model unloaded.")

## Step 2 — Evaluate SFT model (base + LoRA adapter)

In [ ]:
print("Loading SFT model (base + LoRA adapter)...")
sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_REPO,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(sft_model)

print("Running SFT model eval...")
sft_metrics = eval_model(sft_model, sft_tokenizer, EVAL_SET,
                         max_new_tokens=MAX_NEW_TOKENS)

print("\n=== SFT MODEL ===")
for k, v in sft_metrics.items():
    if k == "_per_row":
        continue
    print(f"  {k:<30} {100*v:6.1f}%" if "pct" in k else f"  {k:<30} {v:.3f}")

del sft_model, sft_tokenizer
gc.collect(); torch.cuda.empty_cache()
print("\nSFT model unloaded.")

## Side-by-side summary table

In [ ]:
METRIC_KEYS = [
    "format_pct", "format_after_extract_pct", "exact_pct",
    "service_pct", "operation_pct", "avg_latency", "avg_len",
]

print(f"{'Metric':<30} {'Base':>10} {'SFT':>10} {'Delta':>12}")
print("-" * 64)
for k in METRIC_KEYS:
    b = base_metrics[k]
    s = sft_metrics[k]
    if "pct" in k:
        print(f"{k:<30} {100*b:9.1f}% {100*s:9.1f}% {100*(s-b):+11.1f}pt")
    else:
        print(f"{k:<30} {b:10.3f} {s:10.3f} {s-b:+12.3f}")

## Matplotlib comparison charts

Four views of the same data:
1. **Grouped bar — accuracy metrics** (format, exact, service, operation)
2. **Horizontal bar — latency & length**
3. **Delta chart** — improvement gained by SFT
4. **Radar / spider chart** — overall capability profile

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── Palette & global style ──────────────────────────────────────────────────
COLOR_BASE = "#4C72B0"   # muted blue
COLOR_SFT  = "#DD8452"   # muted orange
COLOR_POS  = "#55A868"   # green  (positive delta)
COLOR_NEG  = "#C44E52"   # red    (negative delta)

plt.rcParams.update({
    "figure.dpi":        120,
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.35,
    "legend.framealpha": 0.9,
    "legend.fontsize":   10,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
})

# ── Data prep ──────────────────────────────────────────────────────────────
acc_keys   = ["format_pct", "format_after_extract_pct", "exact_pct",
              "service_pct", "operation_pct"]
acc_labels = ["Format", "Format\n(extracted)", "Exact Match",
              "Service", "Operation"]

base_acc = [base_metrics[k] * 100 for k in acc_keys]
sft_acc  = [sft_metrics[k]  * 100 for k in acc_keys]
delta_acc = [s - b for s, b in zip(sft_acc, base_acc)]

lat_labels = ["Avg Latency (s)", "Avg Resp Len (chars)"]
base_lat   = [base_metrics["avg_latency"], base_metrics["avg_len"]]
sft_lat    = [sft_metrics["avg_latency"],  sft_metrics["avg_len"]]

# ── Figure layout ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs  = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

ax1 = fig.add_subplot(gs[0, 0])   # grouped bar — accuracy
ax2 = fig.add_subplot(gs[0, 1])   # horizontal bar — latency/length
ax3 = fig.add_subplot(gs[1, 0])   # delta bar
ax4 = fig.add_subplot(gs[1, 1], polar=True)  # radar


# ── 1. Grouped bar — accuracy metrics ─────────────────────────────────────
x      = np.arange(len(acc_labels))
width  = 0.35

bars_b = ax1.bar(x - width/2, base_acc, width,
                 color=COLOR_BASE, label="Base model",
                 edgecolor="white", linewidth=0.6)
bars_s = ax1.bar(x + width/2, sft_acc,  width,
                 color=COLOR_SFT,  label="SFT model",
                 edgecolor="white", linewidth=0.6)

for bar in bars_b:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.8,
             f"{h:.1f}%", ha="center", va="bottom",
             fontsize=8, color=COLOR_BASE, fontweight="bold")
for bar in bars_s:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.8,
             f"{h:.1f}%", ha="center", va="bottom",
             fontsize=8, color=COLOR_SFT, fontweight="bold")

ax1.set_title("Accuracy Metrics — Base vs SFT")
ax1.set_ylabel("Score (%)")
ax1.set_xlabel("Metric")
ax1.set_xticks(x)
ax1.set_xticklabels(acc_labels)
ax1.set_ylim(0, 115)
ax1.legend(loc="upper right")
ax1.yaxis.grid(True, linestyle="--", alpha=0.4)
ax1.set_axisbelow(True)


# ── 2. Horizontal bar — latency & response length ─────────────────────────
y      = np.arange(len(lat_labels))
height = 0.35

hbars_b = ax2.barh(y + height/2, base_lat, height,
                   color=COLOR_BASE, label="Base model",
                   edgecolor="white", linewidth=0.6)
hbars_s = ax2.barh(y - height/2, sft_lat,  height,
                   color=COLOR_SFT,  label="SFT model",
                   edgecolor="white", linewidth=0.6)

for bar, val in zip(hbars_b, base_lat):
    ax2.text(val + max(base_lat + sft_lat) * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f"{val:.2f}", va="center", fontsize=9,
             color=COLOR_BASE, fontweight="bold")
for bar, val in zip(hbars_s, sft_lat):
    ax2.text(val + max(base_lat + sft_lat) * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f"{val:.2f}", va="center", fontsize=9,
             color=COLOR_SFT, fontweight="bold")

ax2.set_title("Latency & Response Length")
ax2.set_xlabel("Value")
ax2.set_yticks(y)
ax2.set_yticklabels(lat_labels)
ax2.legend(loc="lower right")
ax2.xaxis.grid(True, linestyle="--", alpha=0.4)
ax2.set_axisbelow(True)


# ── 3. Delta bar — improvement from SFT ───────────────────────────────────
colors_d = [COLOR_POS if d >= 0 else COLOR_NEG for d in delta_acc]
bars_d   = ax3.bar(x, delta_acc, width=0.5,
                   color=colors_d, edgecolor="white", linewidth=0.6)

for bar, d in zip(bars_d, delta_acc):
    offset = 0.4 if d >= 0 else -1.2
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + offset,
             f"{d:+.1f}pt", ha="center", va="bottom",
             fontsize=9, fontweight="bold",
             color=COLOR_POS if d >= 0 else COLOR_NEG)

ax3.axhline(0, color="#333333", linewidth=0.9, linestyle="-")
ax3.set_title("Delta: SFT − Base (percentage points)")
ax3.set_ylabel("Δ Score (pp)")
ax3.set_xlabel("Metric")
ax3.set_xticks(x)
ax3.set_xticklabels(acc_labels)

pos_patch = mpatches.Patch(color=COLOR_POS, label="Improvement")
neg_patch = mpatches.Patch(color=COLOR_NEG, label="Regression")
ax3.legend(handles=[pos_patch, neg_patch], loc="upper right")
ax3.yaxis.grid(True, linestyle="--", alpha=0.4)
ax3.set_axisbelow(True)


# ── 4. Radar / spider chart ────────────────────────────────────────────────
radar_labels = ["Format", "Format\n(ext.)", "Exact", "Service", "Operation"]
N = len(radar_labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close polygon

base_vals = base_acc + base_acc[:1]
sft_vals  = sft_acc  + sft_acc[:1]

ax4.set_theta_offset(np.pi / 2)
ax4.set_theta_direction(-1)
ax4.set_thetagrids(np.degrees(angles[:-1]), radar_labels, fontsize=9)

ax4.plot(angles, base_vals, color=COLOR_BASE, linewidth=2.0, linestyle="-",
         label="Base model")
ax4.fill(angles, base_vals, color=COLOR_BASE, alpha=0.15)

ax4.plot(angles, sft_vals, color=COLOR_SFT, linewidth=2.0, linestyle="-",
         label="SFT model")
ax4.fill(angles, sft_vals, color=COLOR_SFT, alpha=0.15)

ax4.set_ylim(0, 100)
ax4.set_yticks([20, 40, 60, 80, 100])
ax4.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], fontsize=7)
ax4.yaxis.grid(True, linestyle="--", alpha=0.4)
ax4.xaxis.grid(True, linestyle="--", alpha=0.4)
ax4.set_title("Capability Profile (Radar)",
              pad=20, fontsize=13, fontweight="bold")
ax4.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15))


# ── Super-title & save ─────────────────────────────────────────────────────
fig.suptitle(
    f"Base Model vs SFT Model — AWS RL Eval ({len(EVAL_SET)} prompts)",
    fontsize=15, fontweight="bold", y=1.01,
)

plt.savefig("base_vs_sft_comparison.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Saved → base_vs_sft_comparison.png")

## Bonus — Per-difficulty breakdown

Splits accuracy by `difficulty` tag to see where SFT helps most.

In [ ]:
from collections import defaultdict

def per_difficulty_accuracy(per_row, eval_set, key="operation"):
    buckets = defaultdict(list)
    for row_data, eval_row in zip(per_row, eval_set):
        diff = eval_row.get("difficulty", "unknown")
        buckets[diff].append(float(row_data[key]))
    return {d: sum(v) / len(v) * 100 for d, v in sorted(buckets.items())}

base_by_diff = per_difficulty_accuracy(base_metrics["_per_row"], EVAL_SET)
sft_by_diff  = per_difficulty_accuracy(sft_metrics["_per_row"],  EVAL_SET)

difficulties = list(base_by_diff.keys())
x2    = np.arange(len(difficulties))
width = 0.35

fig2, ax = plt.subplots(figsize=(10, 5))

bb = ax.bar(x2 - width/2,
            [base_by_diff[d] for d in difficulties],
            width, color=COLOR_BASE, label="Base model",
            edgecolor="white", linewidth=0.6)
sb = ax.bar(x2 + width/2,
            [sft_by_diff[d]  for d in difficulties],
            width, color=COLOR_SFT,  label="SFT model",
            edgecolor="white", linewidth=0.6)

for bar in list(bb) + list(sb):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.8,
            f"{h:.0f}%", ha="center", va="bottom", fontsize=9)

ax.set_title("Operation Accuracy by Difficulty — Base vs SFT",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Operation Accuracy (%)")
ax.set_xlabel("Difficulty Tier")
ax.set_xticks(x2)
ax.set_xticklabels([d.capitalize() for d in difficulties])
ax.set_ylim(0, 115)
ax.legend()
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("base_vs_sft_by_difficulty.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Saved → base_vs_sft_by_difficulty.png")

## Qualitative sample — a few predictions side by side

In [ ]:
base_rows = base_metrics["_per_row"]
sft_rows  = sft_metrics["_per_row"]

print(f"{'#':<4} {'Expected':<55} {'Base':<55} {'SFT':<55}")
print("-" * 169)
for i, (br, sr) in enumerate(zip(base_rows[:8], sft_rows[:8])):
    exp  = br["expected"][:52].ljust(55)
    base = br["completion"].strip()[:52].ljust(55)
    sft  = sr["completion"].strip()[:52].ljust(55)
    ok_b = "✓" if br["operation"] else "✗"
    ok_s = "✓" if sr["operation"] else "✗"
    print(f"{i:<4} {exp} {ok_b} {base} {ok_s} {sft}")